#### TiO₂ Polymorphs -- XRD Recovery (Phase 2 model)
- Extension of the CHILI-100K study using the same trained model
- Tests XRD-guided recovery of the three TiO₂ polymorphs (rutile, anatase, brookite) using experimental XRD inputs

In [ ]:
import __init__

In [ ]:
!python _utils/_preprocessing/_process_exp_XRD_inputs.py \
    --input_data 'notebooks/exp_xrd/Rutile.csv' \
    --output_csv 'notebooks/exp_xrd/Rutile_proc.csv'

> Did this for all three profiles

In [ ]:
import pandas as pd
df = pd.read_parquet('_artifacts/cod-xrd/amil/amil-TiO2-nosc-nosg_ref_prompts.parquet')
df.head()

### Generating

> Note: In the load_and_generate.py script, the default context length (how many tokens we can generate to) is set as a global python variable to 1024. We need to change it to 1536 for this study because of this model's extended context

In [ ]:
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_Chili100K-XRD" \
    --reduced_formula_list "TiO2,TiO2,TiO2" \
    --z_list "2,4,8" \
    --xrd_files "notebooks/exp_xrd/Rutile_proc.csv" "notebooks/exp_xrd/Anatase_proc.csv" "notebooks/exp_xrd/Brookite_proc.csv" \
    --num_return_sequences 10 \
    --max_return_attempts 5 \
    --target_valid_cifs 20 \
    --scoring_mode "logp" \
    --temperature 1.1 \
    --output_parquet "_artifacts/tio2-xrd/TiO2-nosc-nosg_gen.parquet"

In [ ]:
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_Chili100K-XRD" \
    --reduced_formula_list "TiO2,TiO2,TiO2" \
    --spacegroups "P4_2/mnm,I4_1/amd,Pbca" \
    --z_list "2,4,8" \
    --xrd_files "notebooks/exp_xrd/Rutile_proc.csv" "notebooks/exp_xrd/Anatase_proc.csv" "notebooks/exp_xrd/Brookite_proc.csv" \
    --level "level_4" \
    --num_return_sequences 10 \
    --max_return_attempts 5 \
    --target_valid_cifs 20 \
    --scoring_mode "logp" \
    --temperature 1.1 \
    --output_parquet "_artifacts/tio2-xrd/TiO2-nosc-sg_gen.parquet"

In [ ]:
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_Chili100K-XRD" \
    --reduced_formula_list "TiO2,TiO2,TiO2" \
    --z_list "4,8,16" \
    --xrd_files "notebooks/exp_xrd/Rutile_proc.csv" "notebooks/exp_xrd/Anatase_proc.csv" "notebooks/exp_xrd/Brookite_proc.csv" \
    --num_return_sequences 10 \
    --max_return_attempts 5 \
    --target_valid_cifs 20 \
    --scoring_mode "logp" \
    --temperature 1.1 \
    --output_parquet "_artifacts/tio2-xrd/TiO2-sc-nosg_gen.parquet"

In [ ]:
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_Chili100K-XRD" \
    --reduced_formula_list "TiO2,TiO2,TiO2" \
    --spacegroups "P4_2/mnm,I4_1/amd,Pbca" \
    --z_list "4,8,16" \
    --xrd_files "notebooks/exp_xrd/Rutile_proc.csv" "notebooks/exp_xrd/Anatase_proc.csv" "notebooks/exp_xrd/Brookite_proc.csv" \
    --level "level_4" \
    --num_return_sequences 10 \
    --max_return_attempts 5 \
    --target_valid_cifs 20 \
    --scoring_mode "logp" \
    --temperature 1.1 \
    --output_parquet "_artifacts/tio2-xrd/TiO2-sc-sg_gen.parquet"

### Post-processing
Patch Material IDs to the known mp- entries (rutile -> mp-2657, anatase -> mp-390, brookite -> mp-1840).

In [ ]:
import pandas as pd

z_sets = ['nosc', 'sc']
sg_modes = ['nosg', 'sg']

for zset in z_sets:
    for sgmode in sg_modes:
        path = f'_artifacts/tio2-xrd/TiO2-{zset}-{sgmode}_gen.parquet'
        df = pd.read_parquet(path).reset_index(drop=True)
        df.loc[0:19, 'Material ID'] = 'mp-2657'
        df.loc[20:39, 'Material ID'] = 'mp-390'
        df.loc[40:59, 'Material ID'] = 'mp-1840'
        df.to_parquet(path, index=False)
        print(f'patched: {path}')

### Metrics

In [ ]:
ref_parquet = '_artifacts/cod-xrd/amil/amil-TiO2-nosc-nosg_ref_prompts.parquet'

for zset in z_sets:
    for sgmode in sg_modes:
        gen = f'_artifacts/tio2-xrd/TiO2-{zset}-{sgmode}_gen.parquet'
        out = f'_artifacts/tio2-xrd/TiO2-{zset}-{sgmode}_metrics.parquet'
        !python _utils/_metrics/XRD_metrics.py \
            --input_parquet {gen} \
            --num_gens 20 \
            --ref_parquet {ref_parquet} \
            --output_parquet {out} \
            --num_workers 4 \
            --validity_check 'none'

In [ ]:
results = {}
for zset in z_sets:
    for sgmode in sg_modes:
        key = f'{zset}-{sgmode}'
        results[key] = pd.read_parquet(f'_artifacts/tio2-xrd/TiO2-{key}_metrics.parquet')

for key, df in results.items():
    print(f'\n{key}')
    display(df)

### Compare to Mattergen_XRD (Phase 1)

In [ ]:
import __init__

In [ ]:
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_Mattergen-XRD" \
    --reduced_formula_list "TiO2,TiO2,TiO2" \
    --z_list "2,4,8" \
    --xrd_files "notebooks/exp_xrd/Rutile_proc.csv" "notebooks/exp_xrd/Anatase_proc.csv" "notebooks/exp_xrd/Brookite_proc.csv" \
    --num_return_sequences 10 \
    --max_return_attempts 5 \
    --target_valid_cifs 20 \
    --scoring_mode "logp" \
    --temperature 1.1 \
    --output_parquet "_artifacts/tio2-xrd/TiO2-nosc-nosg-mg_gen.parquet"

In [ ]:
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_Mattergen-XRD" \
    --reduced_formula_list "TiO2,TiO2,TiO2" \
    --spacegroups "P4_2/mnm,I4_1/amd,Pbca" \
    --z_list "2,4,8" \
    --xrd_files "notebooks/exp_xrd/Rutile_proc.csv" "notebooks/exp_xrd/Anatase_proc.csv" "notebooks/exp_xrd/Brookite_proc.csv" \
    --level "level_4" \
    --num_return_sequences 10 \
    --max_return_attempts 5 \
    --target_valid_cifs 20 \
    --scoring_mode "logp" \
    --temperature 1.1 \
    --output_parquet "_artifacts/tio2-xrd/TiO2-nosc-sg-mg_gen.parquet"

In [ ]:
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_Mattergen-XRD" \
    --reduced_formula_list "TiO2,TiO2,TiO2" \
    --z_list "4,8,16" \
    --xrd_files "notebooks/exp_xrd/Rutile_proc.csv" "notebooks/exp_xrd/Anatase_proc.csv" "notebooks/exp_xrd/Brookite_proc.csv" \
    --num_return_sequences 10 \
    --max_return_attempts 5 \
    --target_valid_cifs 20 \
    --scoring_mode "logp" \
    --temperature 1.1 \
    --output_parquet "_artifacts/tio2-xrd/TiO2-sc-nosg-mg_gen.parquet"

In [ ]:
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_Mattergen-XRD" \
    --reduced_formula_list "TiO2,TiO2,TiO2" \
    --spacegroups "P4_2/mnm,I4_1/amd,Pbca" \
    --z_list "4,8,16" \
    --xrd_files "notebooks/exp_xrd/Rutile_proc.csv" "notebooks/exp_xrd/Anatase_proc.csv" "notebooks/exp_xrd/Brookite_proc.csv" \
    --level "level_4" \
    --num_return_sequences 10 \
    --max_return_attempts 5 \
    --target_valid_cifs 20 \
    --scoring_mode "logp" \
    --temperature 1.1 \
    --output_parquet "_artifacts/tio2-xrd/TiO2-sc-sg-mg_gen.parquet"

### Post-processing
Patch Material IDs to the known mp- entries (rutile -> mp-2657, anatase -> mp-390, brookite -> mp-1840).

In [ ]:
import pandas as pd

z_sets = ['nosc', 'sc']
sg_modes = ['nosg', 'sg']

for zset in z_sets:
    for sgmode in sg_modes:
        path = f'_artifacts/tio2-xrd/TiO2-{zset}-{sgmode}-mg_gen.parquet'
        df = pd.read_parquet(path).reset_index(drop=True)
        df.loc[0:19, 'Material ID'] = 'mp-2657'
        df.loc[20:39, 'Material ID'] = 'mp-390'
        df.loc[40:59, 'Material ID'] = 'mp-1840'
        df.to_parquet(path, index=False)
        print(f'patched: {path}')

### Metrics

In [ ]:
ref_parquet = '_artifacts/cod-xrd/amil/amil-TiO2-nosc-nosg_ref_prompts.parquet'

for zset in z_sets:
    for sgmode in sg_modes:
        gen = f'_artifacts/tio2-xrd/TiO2-{zset}-{sgmode}-mg_gen.parquet'
        out = f'_artifacts/tio2-xrd/TiO2-{zset}-{sgmode}-mg_metrics.parquet'
        !python _utils/_metrics/XRD_metrics.py \
            --input_parquet {gen} \
            --num_gens 20 \
            --ref_parquet {ref_parquet} \
            --output_parquet {out} \
            --num_workers 4 \
            --validity_check 'none'

In [ ]:
results = {}
for zset in z_sets:
    for sgmode in sg_modes:
        key = f'{zset}-{sgmode}'
        results[key] = pd.read_parquet(f'_artifacts/tio2-xrd/TiO2-{key}-mg_metrics.parquet')

for key, df in results.items():
    print(f'\n{key}')
    display(df)